# 관광지 데이터 정제 (pandas)

전국관광지정보표준데이터(raw)를 받아 정제하는 과정을 단계별로 보여 준다.
정제 규칙 자체는 `tourism_platform/lake/pipeline.py`의 `clean_tourspots`에 있고, 이 노트북은 그 결과를 눈으로 확인하는 용도다.

**먼저** `tourism_platform` 폴더에서 `DATA_GO_KR_SERVICE_KEY`를 설정하고 `python -m tourism_platform.lake.run`으로 raw를 받아 둔다.

In [ ]:
from datetime import date
import pandas as pd
from tourism_platform.lake import pipeline, clean_report

day = clean_report.latest_day('tourist_spots')
print('사용하는 raw 날짜:', day)

## 1. raw 불러오기
API 응답 원본을 표로 바꿔 칸과 값을 살펴본다.

In [ ]:
raw = pd.DataFrame(pipeline.raw_items('tourist_spots', day))
print(raw.shape)
raw[['trrsrtNm','trrsrtSe','rdnmadr','latitude','longitude','aceptncCo','prkplceCo']].head()

## 2. 정제 전 상태 확인
좌표가 비었거나 같은 관광지가 여러 번 들어 있는지 본다.

In [ ]:
coords = raw[['latitude','longitude']].apply(pd.to_numeric, errors='coerce')
no_coords = coords.isna().any(axis=1)
print('좌표 없음:', int(no_coords.sum()))
# 좌표가 있는 행끼리만 중복을 센다(정제는 좌표 없는 행을 먼저 빼고 중복을 센다)
valid = raw[~no_coords].assign(la=coords['latitude'].round(4), lo=coords['longitude'].round(4))
print('이름+좌표 중복:', int(valid.duplicated(['trrsrtNm','la','lo']).sum()))

## 3. 정제 실행과 규칙별 건수
노트북에서 직접 센 값과 `clean_tourspots`가 돌려주는 건수가 같은지 비교한다.

In [ ]:
spots, stats = pipeline.clean_tourspots(pipeline.raw_items('tourist_spots', day))
display(clean_report.rule_table(stats))
print('입력 - 걸러진 건수 = 결과:', clean_report.stats_are_consistent(stats))

## 4. 정제 뒤 비어 있는 칸과 튀는 값

In [ ]:
display(clean_report.missing_table(spots, ['sido','sigungu','capacity','parking']))
display(pd.concat([clean_report.outlier_table(spots,'capacity'), clean_report.outlier_table(spots,'parking')]))

## 5. 시도별 집계 (gold)
MongoDB `aggregate`로 낸 결과와 같은지 `mongo_lab.aggregate_vs_pandas`에서 비교한다.

In [ ]:
df = pd.DataFrame(spots)
df.groupby('sido').agg(관광지수=('name','size'), 수용인원합=('capacity','sum'), 주차면수평균=('parking','mean')).round(1)